In [17]:
import pandas as pd
import numpy as np


nat_gas_2000-2025_Trade_Data (Separate CSVs)

In [22]:
# nat_gas_2000-2025_Trade_Data (Separate CSVs)
folder_name = "nat_gas_2000-2025_Trade_data"
year_list = list(range(2000, 2025)) 
df_check = pd.concat([pd.read_csv(f"{folder_name}/{y}") for y in year_list], ignore_index=True)
df = df_check.copy()

df.head(5)

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20000101,2000,1,200001,600,NaN,NaN,M,...,26000.0,False,0.0,False,4090.0,2950.000,4090.000,0,False,True
1,C,M,20000101,2000,1,200001,600,NaN,NaN,M,...,26000.0,False,0.0,False,4090.0,2950.000,4090.000,0,False,True
2,C,M,20000101,2000,1,200001,600,NaN,NaN,M,...,26000.0,False,0.0,False,4090.0,2950.000,4090.000,0,True,False
3,C,M,20000101,2000,1,200001,600,NaN,NaN,M,...,26000.0,False,0.0,False,4090.0,2950.000,4090.000,0,False,True
4,C,M,20000101,2000,1,200001,579,NaN,NaN,X,...,32000.0,False,0.0,False,0.0,29235.494,29235.494,0,False,True


In [19]:
# Ensure period is datetime
if df["period"].dtype != "datetime64[ns]":
    df["period"] = pd.to_datetime(df["period"].astype(str), format="%Y%m", errors="coerce")

In [23]:
# Makind a standard column list
std_col = [
    "period","refYear","refMonth",
    "reporterCode","reporterDesc",
    "partnerCode","partnerDesc",
    "flowCode","flowDesc",
    "cmdCode","cmdDesc",
    "primaryValue",
    "netWgt",
    "qty","qtyUnitCode","qtyUnitAbbr",
    "altQty","altQtyUnitCode"

]

std_col = [i for i in std_col if i in df.columns]
df1 = df[std_col].copy()

df1.head(5)

,period,refYear,refMonth,reporterCode,reporterDesc,partnerCode,partnerDesc,flowCode,flowDesc,cmdCode,cmdDesc,primaryValue,netWgt,qty,qtyUnitCode,qtyUnitAbbr,altQty,altQtyUnitCode
0,200001,2000,1,600,NaN,0,NaN,M,NaN,271111,NaN,4090.000,26000.0,26000.0,8,NaN,26000.0,8
1,200001,2000,1,600,NaN,0,NaN,M,NaN,271111,NaN,4090.000,26000.0,26000.0,8,NaN,26000.0,8
2,200001,2000,1,600,NaN,76,NaN,M,NaN,271111,NaN,4090.000,26000.0,26000.0,8,NaN,26000.0,8
3,200001,2000,1,600,NaN,76,NaN,M,NaN,271111,NaN,4090.000,26000.0,26000.0,8,NaN,26000.0,8
4,200001,2000,1,579,NaN,0,NaN,X,NaN,271111,NaN,29235.494,32000.0,32000.0,8,NaN,32000.0,8


In [25]:
# Basic Checks
print("Rows:", len(df1))
print("Date range:", df1["period"].min(), "to", df1["period"].max())
print("Unique reporters:", df1["reporterCode"].nunique() if "reporterCode" in df1 else None)
print("Unique partners:", df1["partnerCode"].nunique() if "partnerCode" in df1 else None)

Rows: 491806
Date range: 200001 to 202412
Unique reporters: 157
Unique partners: 223


In [26]:
# Missing Data (overall)
miss_values = (df1.isna().mean() * 100).sort_values(ascending=False)
print("\nMissingness rate (%) — overall:\n", miss_values)

# Missingness by year (To check for rough unit price estimation = netWgt/qty)
if "ref_year" in df1:
    cols_check = [i for i in ["primaryValue","netWgt","qty"] if i in df1.columns]
    miss_by_year = (
    df1.groupby("refYear")[cols_check].mean()
    .mul(100)
    .round(2)
    )
    



Missingness rate (%) — overall:
 partnerDesc       100.000000
reporterDesc      100.000000
qtyUnitAbbr       100.000000
cmdDesc           100.000000
flowDesc          100.000000
netWgt              1.893430
qty                 0.972538
altQty              0.972538
refYear             0.000000
period              0.000000
cmdCode             0.000000
flowCode            0.000000
reporterCode        0.000000
partnerCode         0.000000
refMonth            0.000000
primaryValue        0.000000
qtyUnitCode         0.000000
altQtyUnitCode      0.000000
dtype: float64


In [ ]:
# Quantity unit distribution (to see if qty is comparable)
if "qtyUnitAbbr" in df1:
    print("\nTop qty units:\n", df1["qtyUnitAbbr"].value_counts(dropna=False).head(15))


Top qty units:
 qtyUnitAbbr
NaN    491806
Name: count, dtype: int64


In [ ]:
# How many rows can support unit-value via netWgt?
if "netWgt" in df1 and "primaryValue" in df1:
    usable_netwgt = df1["netWgt"].notna() & (df1["netWgt"] > 0) & df1["primaryValue"].notna()
    print("\nRows usable for unit-value using netWgt:", usable_netwgt.sum(), f"({usable_netwgt.mean()*100:.2f}%)")



Rows usable for unit-value using netWgt: 468477 (95.26%)
